# Lección 4: Transformadores

Práctica de sesión 4. Uso de transformadores preentrenados de Hugging Face, uno por familia: solo codificador, codificador-decodificador y solo decodificador.

---
<font size="3">
Christian Cabrera Jojoa<br>
Assistant Research Professor<br>
Department of Computer Science and Technology<br>
University of Cambridge<br>
chc79@cam.ac.uk
</font>

---
**Curso:** Machine Learning Tutorials<br>
**Departamento del curso:** Continuing-education workshops for civil engineering postgraduate students and teaching staff<br>
**Institución del curso:** Pontificia Universidad Católica del Perú (PUCP)


Este cuaderno usa transformadores ya entrenados, uno de cada familia vista en la sesión: solo codificador, codificador-decodificador y solo decodificador. No programamos ni entrenamos la arquitectura. Descargamos modelos de [Hugging Face](https://huggingface.co/models) con la librería `transformers` y los usamos con textos sobre sismos y daños en edificaciones. Cada bloque describe los datos utilizados, la tarea que se quiere solucionar y por qué esa familia encaja.

Ejecute las celdas de arriba hacia abajo. En cada **Comprobar**, confirme el mensaje `OK` antes de seguir. Cada parte tiene una línea marcada con `# TODO`, donde usted cambia una frase o un parámetro. Ya trae un ejemplo para que el cuaderno corra. Cámbielo, vuelva a ejecutar la celda y anote lo que observa en la celda de texto que sigue.

Todo el cuaderno corre en CPU, no hace falta GPU. La primera vez, cada modelo se descarga desde Hugging Face. Son seis modelos, entre 250 MB y 1 GB cada uno, unos 3 GB en total. En Colab la descarga tarda pocos minutos. Si Colab muestra un aviso sobre `HF_TOKEN`, puede ignorarlo: los modelos de este cuaderno son públicos y no piden cuenta.

---

## Instalación

Colab ya trae `transformers` y `torch`. Esta celda asegura una versión reciente de `transformers` y agrega `sentencepiece` y `sacremoses`, que usa el tokenizador del modelo de traducción.

In [ ]:
%pip install -q "transformers>=4.56" sentencepiece sacremoses

## Importaciones

En el siguiente bloque importamos las librerías que usaremos a lo largo del cuaderno.

In [ ]:
import time  # mide cuánto tarda cada descarga

import matplotlib.pyplot as plt  # dibuja la matriz de parecido y las probabilidades
import torch  # tensores y cálculo sin gradientes
from transformers import AutoModel  # transformador sin capa de salida, para vectores
from transformers import AutoModelForCausalLM  # transformador solo decodificador con su capa de salida
from transformers import AutoModelForSeq2SeqLM  # transformador codificador-decodificador
from transformers import AutoTokenizer  # tokenizador que acompaña a cada modelo
from transformers import pipeline  # atajo que une tokenizador, modelo y lectura de la salida

## Cómo se usa un modelo de Hugging Face

[Hugging Face](https://huggingface.co/models) es un repositorio público de modelos. Cada modelo tiene un nombre, por ejemplo `dccuchile/bert-base-spanish-wwm-cased`, y una ficha que explica con qué datos se entrenó y para qué sirve. Los pasos son siempre los mismos:

1. Buscar un modelo en [huggingface.co/models](https://huggingface.co/models). A la izquierda se puede filtrar por tarea, por ejemplo *Fill-Mask* o *Translation*, y por idioma.
2. Copiar su nombre y leer la ficha: la familia, el idioma y la tarea con que se entrenó.
3. Descargar el tokenizador y el modelo con `from_pretrained(nombre)`. La descarga queda guardada, y la segunda vez se carga desde el disco.
4. Pasar el texto por el tokenizador, que lo convierte en números de token.
5. Pasar esos números por el modelo y leer su salida.

`pipeline(tarea, model=nombre)` hace los pasos 3 a 5 de una sola vez para tareas comunes. Lo usamos en las dos primeras tareas. En el resto hacemos los pasos a mano para ver qué entra y qué sale del modelo.

---

## Parte A: solo codificador

Un transformador solo codificador lee todos los tokens de la frase a la vez, los de antes y los de después de cada palabra. No usa la máscara causal. Se preentrena tapando palabras con `[MASK]` y adivinándolas, como BERT. Por eso encaja en tareas donde hay que entender un texto completo: completar un hueco, clasificar un texto o comparar dos textos.

### Completar una palabra

Los datos son una frase sobre un sismo con una palabra tapada con `[MASK]`. La tarea es la misma con que se preentrenó el modelo: proponer la palabra que falta. Usamos `dccuchile/bert-base-spanish-wwm-cased` (BETO), un BERT entrenado con texto en español. Son unos 440 MB.

In [ ]:
MODELO_MASCARA = "dccuchile/bert-base-spanish-wwm-cased"  # nombre del modelo en Hugging Face
inicio = time.time()  # hora antes de la descarga
completar = pipeline("fill-mask", model=MODELO_MASCARA)  # descarga tokenizador y modelo, y los une
print("modelo listo en", round(time.time() - inicio, 1), "segundos")  # tiempo de descarga y carga

`completar(frase)` devuelve una lista con las cinco palabras más probables. Cada elemento trae la palabra (`token_str`) y su probabilidad (`score`). La función `mostrar_candidatas` las imprime en orden y las devuelve para el **Comprobar**.

In [ ]:
def mostrar_candidatas(frase):
    """Imprime las palabras que el modelo propone para [MASK].

    Args:
        frase: Texto con una sola marca [MASK].

    Returns:
        Lista de diccionarios con la palabra y su probabilidad.
    """
    candidatas = completar(frase)  # cinco propuestas, de más a menos probable
    print(frase)  # la frase de entrada
    for candidata in candidatas:  # recorre las propuestas en orden
        palabra = candidata["token_str"]  # texto de la palabra propuesta
        probabilidad = candidata["score"]  # probabilidad que le da el modelo
        print(f"   {palabra:15} {probabilidad:.3f}")  # una línea por propuesta
    return candidatas  # se usa en el Comprobar

Cambie la frase de la línea `# TODO`. Pruebe a mover `[MASK]` a otra palabra, por ejemplo al material de la vivienda o al lugar del sismo. Use una sola marca `[MASK]` por frase.

In [ ]:
frase_con_mascara = "El sismo de magnitud 7 dañó varios [MASK] en Arequipa."  # TODO: cambie la frase o mueva [MASK]
candidatas = mostrar_candidatas(frase_con_mascara)  # propuestas del modelo para esta frase
candidatas_material = mostrar_candidatas("Muchas casas construidas con [MASK] colapsaron durante el terremoto.")  # el material de la vivienda

*Anote lo que observa.* Ejemplo: en la primera frase el modelo propone edificios, puentes y hospitales, cosas que un sismo puede dañar. En la segunda propone materiales: madera, ladrillos, cemento, piedra y tierra. Usa las palabras de los dos lados de `[MASK]`, porque un codificador ve toda la frase. Las probabilidades son más bajas cuando muchas palabras encajan.

### Clasificar el tono de una noticia

Los datos son titulares cortos sobre sismos. La tarea es clasificar cada titular como positivo, neutral o negativo, por ejemplo para revisar primero las noticias de daños. Usamos `lxyuan/distilbert-base-multilingual-cased-sentiments-student`, un codificador pequeño de la familia BERT, ya ajustado para esta clasificación en varios idiomas. Son unos 540 MB. El modelo responde con las etiquetas en inglés: `positive`, `neutral` y `negative`.

In [ ]:
MODELO_TONO = "lxyuan/distilbert-base-multilingual-cased-sentiments-student"  # nombre del modelo en Hugging Face
clasificar = pipeline("text-classification", model=MODELO_TONO)  # descarga tokenizador y modelo, y los une

`clasificar(titular)` devuelve una lista con un diccionario: la etiqueta más probable (`label`) y su probabilidad (`score`). Guardamos las etiquetas en `tonos`, que usa el **Comprobar**.

In [ ]:
titulares = [  # titulares de ejemplo
    "El puente resistió el sismo sin daños.",  # una buena noticia
    "El sismo derrumbó cientos de viviendas de adobe.",  # una mala noticia
    "Un sismo de magnitud 4,5 se registró frente a la costa de Ica.",  # un dato sin juicio
]
tonos = {}  # etiqueta de cada titular
for titular in titulares:  # recorre los titulares
    resultado = clasificar(titular)[0]  # la etiqueta más probable de este titular
    tonos[titular] = resultado["label"]  # guarda la etiqueta
    print(f"{resultado['label']:9} {resultado['score']:.2f}  {titular}")  # una línea por titular

*Observe.* El titular del derrumbe sale `negative` y el del puente `positive`. El titular que solo da un dato sale `negative` con una probabilidad cercana a 0.5: el modelo no está seguro. Un modelo ajustado con titulares de sismos clasificaría mejor ese caso.

### Comparar frases

Los datos son frases cortas de reportes de inspección. La tarea es medir qué tan parecido es el significado de dos frases, aunque usen palabras distintas. Sirve, por ejemplo, para buscar en un archivo de reportes los que hablan de un mismo daño. Usamos `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, un codificador pequeño entrenado para dar vectores parecidos a frases con el mismo significado. Son unos 470 MB.

Aquí hacemos los pasos a mano. `AutoTokenizer` y `AutoModel` descargan el tokenizador y el modelo. `AutoModel` trae el transformador sin capa de salida: devuelve un vector por token.

In [ ]:
MODELO_FRASES = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # nombre del modelo en Hugging Face
tokenizador_frases = AutoTokenizer.from_pretrained(MODELO_FRASES)  # descarga el tokenizador
modelo_frases = AutoModel.from_pretrained(MODELO_FRASES)  # descarga el codificador
modelo_frases.eval()  # modo de uso, sin entrenamiento
print("parámetros:", modelo_frases.num_parameters())  # tamaño del modelo

`vector_de_frase` convierte una frase en un solo vector. El codificador da un vector por token, y promediamos esos vectores. Luego lo dividimos por su largo, así el producto de dos vectores queda entre -1 y 1. Ese número es el parecido: cerca de 1 si las frases dicen lo mismo y cerca de 0 si no tienen relación.

In [ ]:
def vector_de_frase(frase):
    """Vector de una frase, promedio de los vectores de sus tokens.

    Args:
        frase: Texto a convertir.

    Returns:
        Tensor de una dimensión con largo 1.
    """
    entrada = tokenizador_frases(frase, return_tensors="pt")  # números de token de la frase
    with torch.no_grad():  # solo leemos el modelo, sin gradientes
        salida = modelo_frases(**entrada)  # pasa los tokens por el codificador
    vectores = salida.last_hidden_state[0]  # un vector por token
    promedio = vectores.mean(dim=0)  # promedio de los vectores de los tokens
    return promedio / promedio.norm()  # vector de largo 1


def parecido(frase_a, frase_b):
    """Parecido entre el significado de dos frases.

    Args:
        frase_a: Primera frase.
        frase_b: Segunda frase.

    Returns:
        Número entre -1 y 1.
    """
    vector_a = vector_de_frase(frase_a)  # vector de la primera frase
    vector_b = vector_de_frase(frase_b)  # vector de la segunda frase
    return float(vector_a @ vector_b)  # producto de dos vectores de largo 1

Comparamos cada frase con todas las demás y guardamos los valores en `matriz`, que dibujamos y usa el **Comprobar**. Agregue una frase suya en la línea `# TODO` y vea con cuál se parece más.

In [ ]:
frases = [  # frases de reportes de inspección
    "El sismo derrumbó viviendas de adobe en Arequipa.",  # daño en viviendas
    "Varias casas de adobe colapsaron por el terremoto.",  # el mismo daño, con otras palabras
    "El puente resistió el sismo sin daños.",  # otra estructura, sin daño
    "El precio del cobre subió esta semana.",  # otro tema
    "La columna del colegio presenta grietas diagonales.",  # TODO: cambie esta frase por una suya
]
matriz = []  # una fila de parecidos por frase
for frase_fila in frases:  # recorre las filas
    fila = []  # parecidos de esta frase con todas
    for frase_columna in frases:  # recorre las columnas
        fila.append(parecido(frase_fila, frase_columna))  # parecido de este par
    matriz.append(fila)  # guarda la fila

etiquetas = []  # nombre corto de cada frase para los ejes
for numero in range(len(frases)):  # recorre las frases
    etiquetas.append(f"frase {numero + 1}")  # frase 1, frase 2, ...
plt.imshow(matriz, cmap="Blues", vmin=0, vmax=1)  # cada celda es el parecido de un par
plt.xticks(range(len(frases)), etiquetas, rotation=45)  # nombres de las columnas
plt.yticks(range(len(frases)), etiquetas)  # nombres de las filas
for fila_numero in range(len(frases)):  # recorre las filas
    for columna_numero in range(len(frases)):  # recorre las columnas
        valor = matriz[fila_numero][columna_numero]  # parecido de este par
        plt.text(columna_numero, fila_numero, f"{valor:.2f}", ha="center", va="center")  # escribe el valor
plt.colorbar(label="parecido")  # escala de colores
plt.title("Parecido entre frases")  # título del gráfico
plt.show()  # muestra el gráfico

*Anote lo que observa.* Ejemplo: las frases 1 y 2 tienen un parecido de 0.86 aunque solo comparten «adobe», porque dicen lo mismo. La frase del cobre tiene un parecido cercano a 0 con todas. La frase de las grietas se parece más a las frases de viviendas dañadas, alrededor de 0.2, que a la del cobre.

**Comprobar:**

In [ ]:
assert len(candidatas) == 5  # el modelo propone cinco palabras
assert candidatas[0]["score"] >= candidatas[-1]["score"]  # vienen de más a menos probable
assert tonos["El sismo derrumbó cientos de viviendas de adobe."] == "negative"  # el derrumbe es una mala noticia
assert matriz[0][1] > matriz[0][3]  # las dos frases de adobe se parecen más que adobe y cobre
print("Los tres usos del codificador funcionan. OK")

---

## Parte B: codificador-decodificador

Un transformador codificador-decodificador tiene dos partes. El codificador lee todo el texto de entrada, como en la parte A. El decodificador escribe el texto de salida token por token, con la máscara causal, y en cada paso mira los vectores del codificador. Encaja cuando la entrada y la salida son textos distintos: traducir de un idioma a otro o resumir un texto largo. La salida siempre depende del texto de entrada. En la sesión 5 veremos modelos que escriben texto libre.

### Traducir

Los datos son un reporte corto en español sobre un sismo en el sur del Perú. La tarea es traducirlo al inglés, por ejemplo para compartirlo con un equipo internacional o para usarlo con un modelo que solo lee inglés, como el del resumen que sigue. Usamos `Helsinki-NLP/opus-mt-es-en`, un modelo MarianMT entrenado solo para traducir de español a inglés. Son unos 310 MB.

`AutoModelForSeq2SeqLM` descarga un codificador-decodificador con su capa de salida.

In [ ]:
MODELO_TRADUCCION = "Helsinki-NLP/opus-mt-es-en"  # nombre del modelo en Hugging Face
tokenizador_traduccion = AutoTokenizer.from_pretrained(MODELO_TRADUCCION)  # descarga el tokenizador
modelo_traduccion = AutoModelForSeq2SeqLM.from_pretrained(MODELO_TRADUCCION)  # descarga el codificador-decodificador
modelo_traduccion.eval()  # modo de uso, sin entrenamiento
print("parámetros:", modelo_traduccion.num_parameters())  # tamaño del modelo

`traducir` sigue los pasos a mano. El tokenizador convierte la frase en números. `generate` pasa esos números por el codificador y luego llama al decodificador una vez por token de salida, hasta que escribe el token de fin o llega a `max_length` tokens. `decode` vuelve a convertir los números de salida en texto.

In [ ]:
def traducir(frase):
    """Traduce una frase de español a inglés.

    Args:
        frase: Texto en español.

    Returns:
        Texto en inglés.
    """
    entrada = tokenizador_traduccion(frase, return_tensors="pt")  # números de token de la frase
    with torch.no_grad():  # solo leemos el modelo, sin gradientes
        salida = modelo_traduccion.generate(**entrada, max_length=100)  # codifica y decodifica token por token
    return tokenizador_traduccion.decode(salida[0], skip_special_tokens=True)  # números de salida a texto

Traducimos el reporte frase por frase y unimos las traducciones en `reporte_en`, que usa el resumen.

In [ ]:
reporte = [  # reporte en español, una frase por elemento
    "Un sismo de magnitud 6,1 sacudió la región de Arequipa, en el sur del Perú, el domingo.",
    "El movimiento dañó decenas de viviendas de adobe y agrietó los muros de un colegio.",
    "Los ingenieros inspeccionaron dos puentes y no encontraron daños estructurales.",
    "Las autoridades no reportaron fallecidos y abrieron albergues para las familias cuyas casas no eran seguras.",
]
traducciones = []  # frases traducidas, en orden
for frase in reporte:  # recorre el reporte
    traduccion = traducir(frase)  # traduce esta frase
    traducciones.append(traduccion)  # la guarda
    print(frase)  # frase original
    print("  ->", traduccion)  # su traducción
reporte_en = " ".join(traducciones)  # el reporte completo en inglés

### Resumir

Los datos son el reporte traducido. La tarea es resumirlo en pocas palabras, por ejemplo para un titular o una alerta. Usamos `google-t5/t5-small`, el T5 más pequeño, unos 240 MB. T5 se entrenó con muchas tareas de texto a texto, en inglés, y elige la tarea por el comienzo de la entrada: `summarize:` pide un resumen. Por eso resumimos el reporte en inglés.

In [ ]:
MODELO_RESUMEN = "google-t5/t5-small"  # nombre del modelo en Hugging Face
tokenizador_resumen = AutoTokenizer.from_pretrained(MODELO_RESUMEN)  # descarga el tokenizador
modelo_resumen = AutoModelForSeq2SeqLM.from_pretrained(MODELO_RESUMEN)  # descarga el codificador-decodificador
modelo_resumen.eval()  # modo de uso, sin entrenamiento
print("parámetros:", modelo_resumen.num_parameters())  # tamaño del modelo

`resumir` hace los mismos pasos que `traducir`. `largo_maximo` es el máximo de tokens del resumen. `num_beams=4` sigue cuatro resúmenes posibles a la vez y se queda con el más probable.

In [ ]:
def resumir(texto, largo_maximo):
    """Resume un texto en inglés.

    Args:
        texto: Texto en inglés.
        largo_maximo: Máximo de tokens del resumen.

    Returns:
        El resumen.
    """
    entrada = tokenizador_resumen("summarize: " + texto, return_tensors="pt")  # pide la tarea de resumen
    with torch.no_grad():  # solo leemos el modelo, sin gradientes
        salida = modelo_resumen.generate(**entrada, max_length=largo_maximo, num_beams=4)  # escribe el resumen
    return tokenizador_resumen.decode(salida[0], skip_special_tokens=True)  # números de salida a texto

Cambie `LARGO_MAXIMO` en la línea `# TODO`. Pruebe 15 y 30 y compare los resúmenes. Guardamos el resumen en `resumen`, que usa el **Comprobar**.

In [ ]:
LARGO_MAXIMO = 60  # TODO: pruebe 15 y 30
resumen = resumir(reporte_en, LARGO_MAXIMO)  # resumen del reporte traducido
print("reporte:", len(reporte_en.split()), "palabras")  # largo del reporte
print("resumen:", len(resumen.split()), "palabras")  # largo del resumen
print(resumen)  # el resumen

*Anote lo que observa.* Ejemplo: con 60 tokens el resumen copia tres de las cuatro frases del reporte y deja fuera la primera, la del lugar y la fecha. Con 30 y con 15 se corta a media frase. T5-small resume eligiendo y copiando partes del texto de entrada, no escribe ideas nuevas. También pasa todo a minúsculas.

**Comprobar:**

In [ ]:
assert len(traducciones) == len(reporte)  # una traducción por frase
assert "earthquake" in reporte_en.lower()  # la palabra sismo aparece traducida
assert len(resumen.split()) < len(reporte_en.split())  # el resumen es más corto que el reporte
print("El codificador-decodificador traduce y resume. OK")

---

## Parte C: solo decodificador

Un transformador solo decodificador lee los tokens de izquierda a derecha, con la máscara causal: cada token mira solo los anteriores. Se preentrena con la tarea de la sesión, predecir el siguiente token, como GPT. Encaja cuando queremos saber cómo sigue un texto. Aquí solo miramos las probabilidades del siguiente token. No dejamos que el modelo escriba texto, eso lo veremos en la sesión 5.

Los datos son el comienzo de frases sobre sismos. La tarea es ver qué tokens el modelo considera más probables a continuación. Usamos `Qwen/Qwen2.5-0.5B`, un modelo abierto pequeño que maneja español, unos 1 GB. `AutoModelForCausalLM` descarga el transformador con su capa de salida, la que da un puntaje por cada token del vocabulario. `dtype=torch.float32` pide los pesos con la precisión habitual de la CPU, así los resultados no cambian entre versiones de la librería.

In [ ]:
MODELO_SIGUIENTE = "Qwen/Qwen2.5-0.5B"  # nombre del modelo en Hugging Face
tokenizador_siguiente = AutoTokenizer.from_pretrained(MODELO_SIGUIENTE)  # descarga el tokenizador
modelo_siguiente = AutoModelForCausalLM.from_pretrained(MODELO_SIGUIENTE, dtype=torch.float32)  # descarga el decodificador
modelo_siguiente.eval()  # modo de uso, sin entrenamiento
print("parámetros:", modelo_siguiente.num_parameters())  # tamaño del modelo
print("tokens del vocabulario:", len(tokenizador_siguiente))  # cuántos tokens puede proponer

`siguientes_tokens` sigue los pasos de la sesión. El modelo da un puntaje por token del vocabulario en cada posición. Tomamos los de la última posición, los pasamos por softmax para tener probabilidades y nos quedamos con las `k` más altas.

In [ ]:
def siguientes_tokens(contexto, k):
    """Los k tokens más probables después de un texto.

    Args:
        contexto: Comienzo de una frase.
        k: Cuántos tokens devolver.

    Returns:
        Lista de pares (texto del token, probabilidad), de más a menos probable.
    """
    entrada = tokenizador_siguiente(contexto, return_tensors="pt")  # números de token del contexto
    with torch.no_grad():  # solo leemos el modelo, sin gradientes
        salida = modelo_siguiente(**entrada)  # un puntaje por token del vocabulario en cada posición
    puntajes = salida.logits[0, -1]  # puntajes de la última posición
    probabilidades = torch.softmax(puntajes, dim=-1)  # puntajes a probabilidades que suman 1
    mejores = torch.topk(probabilidades, k)  # las k probabilidades más altas
    pares = []  # resultado
    for probabilidad, numero in zip(mejores.values, mejores.indices):  # recorre los k mejores
        texto_token = tokenizador_siguiente.decode([int(numero)])  # texto del token
        pares.append((texto_token, float(probabilidad)))  # guarda el par
    return pares  # se usa en el gráfico y en el Comprobar

`dibujar_siguientes` muestra los pares en un gráfico de barras. Un espacio al comienzo de un token es parte del token.

In [ ]:
def dibujar_siguientes(contexto, pares):
    """Gráfico de barras con las probabilidades del siguiente token.

    Args:
        contexto: Comienzo de la frase, para el título.
        pares: Lista de (texto del token, probabilidad).
    """
    textos = []  # texto de cada barra
    valores = []  # alto de cada barra
    for texto_token, probabilidad in pares:  # recorre los pares
        textos.insert(0, repr(texto_token))  # al comienzo, así el más probable queda arriba; repr muestra los espacios
        valores.insert(0, probabilidad)  # probabilidad del token, en el mismo orden
    plt.barh(textos, valores)  # una barra por token
    plt.xlabel("probabilidad")  # nombre del eje
    plt.title(contexto + " ...")  # el contexto como título
    plt.show()  # muestra el gráfico

Cambie el contexto en la línea `# TODO`. Pruebe a agregar o quitar palabras, por ejemplo el lugar o el material, y vea cómo cambian las probabilidades. Cambie también `K`. Guardamos los pares en `pares`, que usa el **Comprobar**.

In [ ]:
contexto = "Después del sismo, los ingenieros revisaron los daños en el"  # TODO: cambie el contexto
K = 5  # cuántos tokens mostrar
pares = siguientes_tokens(contexto, K)  # los K tokens más probables
for texto_token, probabilidad in pares:  # recorre los pares
    print(f"{texto_token!r:15} {probabilidad:.3f}")  # una línea por token
dibujar_siguientes(contexto, pares)  # gráfico de barras

*Anote lo que observa.* Ejemplo: el token más probable es ` ed`, el comienzo de ` edificio`. El modelo propone tokens, piezas de palabra, no palabras completas. Las cinco probabilidades suman menos de 0.2, porque el resto se reparte entre los más de 150 mil tokens del vocabulario. Pruebe también un contexto que termine a mitad de palabra, por ejemplo `El sismo de magn`.

### Cada posición predice la siguiente

En el entrenamiento, cada posición de una frase predice su siguiente token, todas a la vez. Lo vemos con una frase completa: para cada posición mostramos el token que sigue en la frase, la probabilidad que el modelo le daba y el token que el modelo creía más probable. Es la misma lectura de probabilidades de arriba, sin escribir nada nuevo.

In [ ]:
frase = "El sismo de magnitud 7 dañó varias viviendas de adobe en Arequipa."  # frase completa
entrada = tokenizador_siguiente(frase, return_tensors="pt")  # números de token de la frase
with torch.no_grad():  # solo leemos el modelo, sin gradientes
    salida = modelo_siguiente(**entrada)  # puntajes en todas las posiciones a la vez
probabilidades = torch.softmax(salida.logits[0], dim=-1)  # probabilidades en cada posición
numeros = entrada["input_ids"][0]  # números de token de la frase
aciertos = 0  # posiciones donde el token real era el más probable
for posicion in range(len(numeros) - 1):  # cada posición menos la última
    real = int(numeros[posicion + 1])  # el token que sigue en la frase
    probable = int(torch.argmax(probabilidades[posicion]))  # el token que el modelo creía más probable
    if real == probable:  # el modelo acertó
        aciertos = aciertos + 1  # cuenta el acierto
    texto_actual = tokenizador_siguiente.decode([int(numeros[posicion])])  # token de esta posición
    texto_real = tokenizador_siguiente.decode([real])  # texto del siguiente token real
    texto_probable = tokenizador_siguiente.decode([probable])  # texto del token más probable
    probabilidad_real = float(probabilidades[posicion, real])  # probabilidad del token real
    print(f"{texto_actual!r:12} -> real {texto_real!r:12} {probabilidad_real:.3f} | más probable {texto_probable!r}")  # una línea por posición
print("aciertos:", aciertos, "de", len(numeros) - 1)  # cuántas posiciones acertó

Donde la probabilidad del token real es baja, el error de entropía cruzada del entrenamiento es alto. El material (` ad`, de adobe) y el lugar (` Are`, de Arequipa) son difíciles de adivinar. Las piezas que terminan una palabra ya empezada, como `itud` o `quip`, son fáciles.

**Comprobar:**

In [ ]:
assert len(pares) == K  # el modelo devolvió K tokens
assert pares[0][1] >= pares[-1][1]  # vienen de más a menos probable
total = 0.0  # suma de las K probabilidades
for texto_token, probabilidad in pares:  # recorre los pares
    total = total + probabilidad  # suma esta probabilidad
assert 0 < total <= 1.0001  # son parte de una distribución que suma 1
print("El decodificador da la probabilidad del siguiente token. OK")

---

## Resumen

| Familia | Modelo | Qué ve | Tareas de este cuaderno |
|---|---|---|---|
| Solo codificador | BETO, DistilBERT multilingüe, MiniLM multilingüe | Todos los tokens de la frase | Completar una palabra, clasificar el tono, comparar frases |
| Codificador-decodificador | MarianMT, T5 | El codificador ve toda la entrada y el decodificador los tokens ya escritos | Traducir y resumir |
| Solo decodificador | Qwen2.5-0.5B | Los tokens anteriores, con la máscara causal | Probabilidad del siguiente token |

En la sesión 5 usaremos modelos solo decodificador mucho más grandes, los modelos de lenguaje grandes, para que escriban texto.